# GAT with SR-GNN-Style Session Readout

This notebook follows the SR-GNN session recommendation pipeline, but replaces the GGNN encoder with a Graph Attention Network. The readout stays SR-GNN-style: last-click local preference plus attention-based global preference.

## Architecture

A session prefix `[v1, v2, ..., vt]` is turned into a graph. Nodes are unique items from the prefix. Directed edges follow observed clicks, for example `v1 -> v2`; reverse edges are also built so the model can read both incoming and outgoing transition context. This iteration restores the historically strongest iteration-1 graph contract from commit `7bff12dc33875b37e78bd8d15ae96963673a899c`: repeated transitions are aggregated into three edge features: normalized transition probability, repeat strength, and recency. These features are passed into `GATConv(edge_dim=3)`.

The encoder is a bidirectional weighted GAT block shared with the TAGNN notebook. It uses the same item embedding table for both directions, but separate GAT weights for forward and backward transitions:

```text
session prefix
     |
directed weighted session graph
     |
item embedding, dim=100
     |
     +--> forward GATConv(edge_dim=3): 4 heads x 25 dims, attention dropout=0.05, feature dropout=0.10
     |
     +--> backward GATConv(edge_dim=3): 4 heads x 25 dims, attention dropout=0.05, feature dropout=0.10
                 |
plain residual update in each direction
                 |
concat forward/backward states, dim=200
                 |
linear direction fusion, 200 -> 100
                 |
contextual node embeddings
                 |
SR-GNN local/global readout
                 |
scores for all items
```

GAT details used here:

- Layer type: `torch_geometric.nn.GATConv`.
- Edge features: transition probability, repeat strength, and recency through `edge_dim=3`.
- Hidden size: `100`.
- Number of GAT layers per direction: `1`.
- Attention heads: `4`.
- Per-head output size: `25`, concatenated back to `100`.
- Default regularization variant: `r5_restored_baseline_label_smoothing` with attention dropout `0.05`, feature dropout `0.10`, and label smoothing `0.01`.
- Activation: `ELU`.
- Residual connection: plain residual addition after the GAT update.
- Self-loops: enabled by `GATConv`.
- Direction fusion: concatenate forward and backward node states, then apply a linear layer `200 -> 100`.
- Encoder output skip: disabled; the post-fusion identity skip from iteration 7 is treated as a failed ablation.

In a GAT layer, a node learns attention weights over its connected items instead of treating all neighbors equally. Passing transition probability, repeat strength, and recency gives the architecture explicit frequency and order signals while preserving the SR-GNN-style forward/backward channels.

After graph encoding, node embeddings are mapped back to the original click order. The last clicked item gives the local preference. Attention over all prefix positions gives the global preference. These two vectors are concatenated and projected into one session representation.

## Prediction

The final session vector is multiplied by the item embedding matrix. This gives one score for every candidate item. Items are ranked by score, and the top 20 are used for Precision@20 and MRR@20.

This Kaggle-ready notebook prepares raw Yoochoose and Diginetica inputs, runs the SR-GNN/TAGNN preprocessing pipeline, and trains the GAT + SR-GNN-style readout model in one file.

Expected Kaggle input directories:
- `/kaggle/input/datasets/chadgostopp/recsys-challenge-2015` containing `yoochoose-clicks.dat`
- `/kaggle/input/datasets/profalbusdumbledore/diginetica-dataset` containing `train-item-views.csv`

## Prepare Datasets

The Kaggle inputs contain the original raw files. This section reads the raw Kaggle files directly and prepares the in-memory prefix-label examples used by the following sections.

In [1]:
import subprocess
import sys

import torch


def run_pip_install(args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


torch_version = torch.__version__.split("+")[0]
cuda_tag = "cpu"
if torch.cuda.is_available() and torch.version.cuda:
    cuda_tag = "cu" + torch.version.cuda.replace(".", "")

print(f"torch={torch.__version__}")
print(f"torch_cuda={torch.version.cuda}")
if torch.cuda.is_available():
    print(f"gpu={torch.cuda.get_device_name(0)}")
    print(f"torch_cuda_arch_list={torch.cuda.get_arch_list()}")

print(f"Installing PyG wheels for torch={torch_version}, cuda={cuda_tag}")
run_pip_install(
    [
        "--force-reinstall",
        "--no-deps",
        "torch_geometric",
        "pyg_lib",
        "torch_scatter",
        "torch_sparse",
        "torch_cluster",
        "torch_spline_conv",
        "-f",
        f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html",
    ]
)


torch=2.10.0+cu128
torch_cuda=12.8
gpu=Tesla T4
torch_cuda_arch_list=['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
Installing PyG wheels for torch=2.10.0, cuda=cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 40.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 85.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 79.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 78.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.5 MB/s eta 0:00:00


In [2]:
import math
import os
import random
import time
from collections import Counter
from pathlib import Path

MPLCONFIGDIR = Path("/tmp/matplotlib")
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GATConv
from torch_geometric.utils import softmax as pyg_softmax


In [3]:
KAGGLE_INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")

YOOCHOOSE_INPUT_DIR = KAGGLE_INPUT_DIR / "datasets/chadgostopp/recsys-challenge-2015"
DIGINETICA_INPUT_DIR = (
    KAGGLE_INPUT_DIR / "datasets/profalbusdumbledore/diginetica-dataset"
)
YOOCHOOSE_SOURCE = YOOCHOOSE_INPUT_DIR / "yoochoose-clicks.dat"
DIGINETICA_SOURCE = DIGINETICA_INPUT_DIR / "train-item-views.csv"

RESULTS_DIR = OUTPUT_DIR / "results"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"

for directory in [RESULTS_DIR, CHECKPOINTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"output_dir={OUTPUT_DIR}")
print(f"Using Yoochoose source: {YOOCHOOSE_SOURCE}")
print(f"Using Diginetica source: {DIGINETICA_SOURCE}")

output_dir=/kaggle/working
Using Yoochoose source: /kaggle/input/datasets/chadgostopp/recsys-challenge-2015/yoochoose-clicks.dat
Using Diginetica source: /kaggle/input/datasets/profalbusdumbledore/diginetica-dataset/train-item-views.csv


## Preprocess Sessions

The preprocessing flow builds ordered sessions, removes short sessions and rare items, splits chronologically, remaps item ids from training data, and expands sessions into prefix-label examples.

In [4]:
def load_yoochoose_sessions(path):
    df = pd.read_csv(
        path,
        header=None,
        usecols=[0, 1, 2],
        names=["session_id", "timestamp", "item_id"],
    )
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.tz_convert(None)

    session_items = {}
    session_dates = {}
    current_session_id = None
    current_timestamp = None

    for row in df.itertuples(index=False):
        session_id = int(row.session_id)
        if current_timestamp is not None and current_session_id != session_id:
            session_dates[current_session_id] = current_timestamp

        current_session_id = session_id
        current_timestamp = row.timestamp

        if session_id in session_items:
            session_items[session_id].append(row.item_id)
        else:
            session_items[session_id] = [row.item_id]

    if current_session_id is not None:
        session_dates[current_session_id] = current_timestamp

    return [
        (session_id, session_dates[session_id], items)
        for session_id, items in session_items.items()
    ]


def load_diginetica_sessions(path):
    df = pd.read_csv(
        path,
        sep=";",
        usecols=["sessionId", "itemId", "timeframe", "eventdate"],
    )
    df = df.rename(columns={"sessionId": "session_id", "itemId": "item_id"})
    df["eventdate"] = pd.to_datetime(df["eventdate"], format="%Y-%m-%d")

    session_clicks = {}
    session_dates = {}
    current_session_id = None
    current_date = None

    for row in df.itertuples(index=False):
        session_id = int(row.session_id)
        if current_date is not None and current_session_id != session_id:
            session_dates[current_session_id] = current_date

        current_session_id = session_id
        current_date = row.eventdate

        click = (row.item_id, int(row.timeframe))
        if session_id in session_clicks:
            session_clicks[session_id].append(click)
        else:
            session_clicks[session_id] = [click]

    if current_session_id is not None:
        session_dates[current_session_id] = current_date

    sessions = []
    for session_id, clicks in session_clicks.items():
        ordered_clicks = sorted(clicks, key=lambda click: click[1])
        items = [item for item, _ in ordered_clicks]
        sessions.append((session_id, session_dates[session_id], items))
    return sessions

In [5]:
def drop_short_sessions(sessions):
    return [session for session in sessions if len(session[2]) >= 2]


def drop_rare_items(sessions, min_freq=5):
    counts = Counter()
    for _, _, items in sessions:
        counts.update(items)

    result = []
    for session_id, date, items in sessions:
        kept = [i for i in items if counts[i] >= min_freq]
        if len(kept) >= 2:
            result.append((session_id, date, kept))
    return result


def sort_by_date(sessions):
    return sorted(sessions, key=lambda session: session[1])


def split_by_date(sessions, test_days):
    max_date = max(date for _, date, _ in sessions)
    split_date = max_date - pd.Timedelta(days=test_days)
    train = [s for s in sessions if s[1] < split_date]
    test = [s for s in sessions if s[1] > split_date]
    return train, test


def renumber_training_items(train_sessions):
    item_to_index = {}
    next_item_index = 1
    remapped_sessions = []

    for session_id, date, items in train_sessions:
        remapped_items = []
        for item in items:
            if item not in item_to_index:
                item_to_index[item] = next_item_index
                next_item_index += 1
            remapped_items.append(item_to_index[item])
        remapped_sessions.append((session_id, date, remapped_items))

    return remapped_sessions, item_to_index


def remap_test_sessions(test_sessions, item_to_index):
    remapped_sessions = []
    for session_id, date, items in test_sessions:
        remapped_items = [
            item_to_index[item] for item in items if item in item_to_index
        ]
        if len(remapped_items) >= 2:
            remapped_sessions.append((session_id, date, remapped_items))
    return remapped_sessions


def expand_sessions(sessions):
    examples = []
    for session_id, date, items in sessions:
        for reverse_offset in range(1, len(items)):
            examples.append(
                (session_id, date, items[:-reverse_offset], items[-reverse_offset])
            )
    return examples


def keep_recent_fraction(examples, denominator):
    if denominator is None:
        return examples
    keep = len(examples) // denominator
    return examples[-keep:] if keep else examples


def prefix_label_rows(examples):
    return [(list(prefix), int(label)) for _, _, prefix, label in examples]


def vocabulary_size_from_rows(*row_groups):
    max_item_id = 0
    for rows in row_groups:
        for prefix, label in rows:
            max_item_id = max(max_item_id, int(label), max(prefix))
    return max_item_id + 1


def preprocess(sessions, test_days, train_fraction_denominator):
    sessions = drop_short_sessions(sessions)
    sessions = drop_rare_items(sessions)
    sessions = sort_by_date(sessions)
    train_sessions, test_sessions = split_by_date(sessions, test_days)

    train_sessions, item_to_index = renumber_training_items(train_sessions)
    test_sessions = remap_test_sessions(test_sessions, item_to_index)

    train_examples = expand_sessions(train_sessions)
    test_examples = expand_sessions(test_sessions)
    train_examples = keep_recent_fraction(train_examples, train_fraction_denominator)
    return train_examples, test_examples

In [6]:
yoochoose_sessions = load_yoochoose_sessions(YOOCHOOSE_SOURCE)
diginetica_sessions = load_diginetica_sessions(DIGINETICA_SOURCE)

yoochoose_1_64_train, yoochoose_1_64_test = preprocess(
    yoochoose_sessions, test_days=1, train_fraction_denominator=64
)
diginetica_train, diginetica_test = preprocess(
    diginetica_sessions, test_days=7, train_fraction_denominator=None
)

YOOCHOOSE_1_64_TRAIN_ROWS = prefix_label_rows(yoochoose_1_64_train)
YOOCHOOSE_1_64_TEST_ROWS = prefix_label_rows(yoochoose_1_64_test)
DIGINETICA_TRAIN_ROWS = prefix_label_rows(diginetica_train)
DIGINETICA_TEST_ROWS = prefix_label_rows(diginetica_test)

print(f"Yoochoose 1/64 train examples: {len(YOOCHOOSE_1_64_TRAIN_ROWS):,}")
print(f"Yoochoose 1/64 test examples: {len(YOOCHOOSE_1_64_TEST_ROWS):,}")
print(f"Diginetica train examples: {len(DIGINETICA_TRAIN_ROWS):,}")
print(f"Diginetica test examples: {len(DIGINETICA_TEST_ROWS):,}")

Yoochoose 1/64 train examples: 369,859
Yoochoose 1/64 test examples: 55,898
Diginetica train examples: 719,470
Diginetica test examples: 60,858


In [7]:
def configure_accelerator():
    """Configure the notebook for the Kaggle NVIDIA T4 CUDA path."""
    if not torch.cuda.is_available():
        print("cuda_device=None")
        return

    device_index = 0
    device_count = torch.cuda.device_count()
    props = torch.cuda.get_device_properties(device_index)
    capability = torch.cuda.get_device_capability(device_index)
    arch = f"sm_{capability[0]}{capability[1]}"
    supported_arches = torch.cuda.get_arch_list()
    if arch not in supported_arches:
        raise RuntimeError(
            f"Installed PyTorch CUDA build does not support {arch}. "
            f"Supported arches: {supported_arches}"
        )

    # T4 is compute capability 7.5. It supports Tensor Cores but not TF32.
    has_tf32 = capability[0] >= 8
    torch.backends.cuda.matmul.allow_tf32 = has_tf32
    torch.backends.cudnn.allow_tf32 = has_tf32
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

    torch.cuda.set_device(device_index)
    torch.cuda.empty_cache()
    print(f"cuda_visible_device_count={device_count}")
    print(f"cuda_device={torch.cuda.get_device_name(device_index)}")
    print(f"cuda_capability={capability[0]}.{capability[1]}")
    print(f"cuda_memory_gb={props.total_memory / 1024**3:.1f}")
    print(f"torch={torch.__version__}")
    print(f"torch_cuda={torch.version.cuda}")
    print(f"tf32_enabled={has_tf32}")
    if device_count > 1:
        print("multi_gpu_note=not using second GPU; notebooks run single-process PyTorch on cuda:0")


configure_accelerator()

print("Available Kaggle input files:")
for dirname, _, filenames in os.walk(KAGGLE_INPUT_DIR):
    for filename in filenames:
        print(Path(dirname) / filename)


cuda_visible_device_count=2
cuda_device=Tesla T4
cuda_capability=7.5
cuda_memory_gb=14.6
torch=2.10.0+cu128
torch_cuda=12.8
tf32_enabled=False
multi_gpu_note=not using second GPU; notebooks run single-process PyTorch on cuda:0
Available Kaggle input files:
/kaggle/input/datasets/profalbusdumbledore/diginetica-dataset/products.csv
/kaggle/input/datasets/profalbusdumbledore/diginetica-dataset/product-categories.csv
/kaggle/input/datasets/profalbusdumbledore/diginetica-dataset/train-queries.csv
/kaggle/input/datasets/profalbusdumbledore/diginetica-dataset/train-item-views.csv
/kaggle/input/datasets/profalbusdumbledore/diginetica-dataset/train-clicks.csv
/kaggle/input/datasets/profalbusdumbledore/diginetica-dataset/train-purchases.csv
/kaggle/input/datasets/chadgostopp/recsys-challenge-2015/yoochoose-buys.dat
/kaggle/input/datasets/chadgostopp/recsys-challenge-2015/yoochoose-clicks.dat
/kaggle/input/datasets/chadgostopp/recsys-challenge-2015/yoochoose-test.dat
/kaggle/input/datasets/chadgo

## Session graph dataset

The graph builder follows the SR-GNN/TAGNN preprocessing convention: one node per unique item in the prefix, directed edges for consecutive clicks, and separate forward/backward transition channels. This iteration keeps duplicate transition aggregation and expands the edge signal used by GAT: transition probability, repeat strength, and recency are passed to `GATConv(edge_dim=3)`. The original scalar transition probabilities are still stored for inspection.

In [8]:
def build_session_graph(prefix, label):
    """Turn a `(prefix, label)` example into a PyG session graph.

    Duplicate transitions are aggregated. Each directed edge receives three GAT
    features: normalized transition probability, repeat strength, and recency.
    """
    unique_items = list(dict.fromkeys(prefix))
    item_to_node = {item: index for index, item in enumerate(unique_items)}
    click_sequence = [item_to_node[item] for item in prefix]
    transitions = list(zip(click_sequence[:-1], click_sequence[1:]))

    edge_counts = Counter(transitions)
    latest_positions = {}
    for position, transition in enumerate(transitions, start=1):
        latest_positions[transition] = position

    out_degree = Counter()
    in_degree = Counter()
    for (source, target), count in edge_counts.items():
        out_degree[source] += count
        in_degree[target] += count

    transition_count = max(1, len(transitions))
    max_count = max(edge_counts.values(), default=1)

    def edge_features(probability, transition, count):
        repeat_strength = math.log1p(count) / math.log1p(max_count)
        recency = latest_positions[transition] / transition_count
        return [probability, repeat_strength, recency]

    forward_sources, forward_targets, forward_weights, forward_attrs = [], [], [], []
    backward_sources, backward_targets, backward_weights, backward_attrs = [], [], [], []
    for transition, count in edge_counts.items():
        source, target = transition
        forward_probability = count / out_degree[source]
        backward_probability = count / in_degree[target]

        forward_sources.append(source)
        forward_targets.append(target)
        forward_weights.append(forward_probability)
        forward_attrs.append(edge_features(forward_probability, transition, count))

        backward_sources.append(target)
        backward_targets.append(source)
        backward_weights.append(backward_probability)
        backward_attrs.append(edge_features(backward_probability, transition, count))

    if forward_sources:
        forward_edge_index = torch.tensor(
            [forward_sources, forward_targets], dtype=torch.long
        )
        forward_edge_weight = torch.tensor(forward_weights, dtype=torch.float)
        forward_edge_attr = torch.tensor(forward_attrs, dtype=torch.float)
        backward_edge_index = torch.tensor(
            [backward_sources, backward_targets], dtype=torch.long
        )
        backward_edge_weight = torch.tensor(backward_weights, dtype=torch.float)
        backward_edge_attr = torch.tensor(backward_attrs, dtype=torch.float)
    else:
        forward_edge_index = torch.zeros((2, 0), dtype=torch.long)
        forward_edge_weight = torch.zeros(0, dtype=torch.float)
        forward_edge_attr = torch.zeros((0, 3), dtype=torch.float)
        backward_edge_index = torch.zeros((2, 0), dtype=torch.long)
        backward_edge_weight = torch.zeros(0, dtype=torch.float)
        backward_edge_attr = torch.zeros((0, 3), dtype=torch.float)

    return Data(
        x=torch.tensor(unique_items, dtype=torch.long),
        edge_index=forward_edge_index,
        edge_weight=forward_edge_weight,
        edge_attr=forward_edge_attr,
        forward_edge_index=forward_edge_index,
        forward_edge_weight=forward_edge_weight,
        forward_edge_attr=forward_edge_attr,
        backward_edge_index=backward_edge_index,
        backward_edge_weight=backward_edge_weight,
        backward_edge_attr=backward_edge_attr,
        sequence=torch.tensor(click_sequence, dtype=torch.long),
        sequence_length=torch.tensor(len(click_sequence), dtype=torch.long),
        last_click=torch.tensor(click_sequence[-1], dtype=torch.long),
        y=torch.tensor(label, dtype=torch.long),
        num_nodes=len(unique_items),
    )


class SessionGraphDataset(Dataset):
    """Lazy dataset of session-prefix graphs.

    Graphs are built on demand so training does not materialize all PyG
    objects in memory before training starts.
    """

    def __init__(self, prefix_label_rows):
        self.prefix_label_rows = prefix_label_rows

    def __len__(self):
        return len(self.prefix_label_rows)

    def __getitem__(self, index):
        prefix, label = self.prefix_label_rows[index]
        return build_session_graph(prefix, label)

## Load datasets and compute vocabulary size

In [9]:
yoochoose_1_64_num_items = vocabulary_size_from_rows(
    YOOCHOOSE_1_64_TRAIN_ROWS, YOOCHOOSE_1_64_TEST_ROWS
)
diginetica_num_items = vocabulary_size_from_rows(
    DIGINETICA_TRAIN_ROWS, DIGINETICA_TEST_ROWS
)

print(f"Yoochoose 1/64 num_items: {yoochoose_1_64_num_items:,}")
print(f"Diginetica      num_items: {diginetica_num_items:,}")

Yoochoose 1/64 num_items: 37,484
Diginetica      num_items: 43,098


## Bidirectional GAT encoder

The original SR-GNN/TAGNN encoders use separate incoming and outgoing transition channels. This experiment keeps the historically strongest weighted `GATConv` base, but expands each edge from one scalar to three features and splits attention dropout from feature dropout. Each directional stack applies `GATConv(edge_dim=3)`, `ELU`, feature dropout, and plain residual addition. It does not use `GATv2Conv`, gated residual blending, per-direction `LayerNorm`, or the iteration-7 post-fusion identity skip.

In [10]:
class DirectionalGATStack(nn.Module):
    """Weighted GAT stack for one transition direction."""

    def __init__(
        self,
        hidden_dim=100,
        num_layers=1,
        num_heads=4,
        attention_dropout=0.05,
        feature_dropout=0.1,
        concat_heads=True,
        residual=True,
    ):
        super().__init__()

        if concat_heads and hidden_dim % num_heads != 0:
            raise ValueError(
                f"hidden_dim ({hidden_dim}) must be divisible by num_heads "
                f"({num_heads}) when concat_heads=True"
            )

        self.feature_dropout = feature_dropout
        self.residual = residual
        per_head_dim = hidden_dim // num_heads if concat_heads else hidden_dim
        self.gat_layers = nn.ModuleList(
            GATConv(
                in_channels=hidden_dim,
                out_channels=per_head_dim,
                heads=num_heads,
                concat=concat_heads,
                dropout=attention_dropout,
                add_self_loops=True,
                edge_dim=3,
            )
            for _ in range(num_layers)
        )

    def forward(self, node_features, edge_index, edge_attr=None):
        for gat_layer in self.gat_layers:
            previous = node_features
            node_features = F.elu(gat_layer(node_features, edge_index, edge_attr))
            node_features = F.dropout(
                node_features, p=self.feature_dropout, training=self.training
            )
            if self.residual:
                node_features = node_features + previous
        return node_features


class BidirectionalGATEncoder(nn.Module):
    """Separate forward/backward weighted GAT stacks with shared embeddings."""

    def __init__(
        self,
        num_items,
        hidden_dim=100,
        num_layers=1,
        num_heads=4,
        attention_dropout=0.05,
        feature_dropout=0.1,
    ):
        super().__init__()
        self.embedding = nn.Embedding(num_items, hidden_dim, padding_idx=0)
        self.forward_gat = DirectionalGATStack(
            hidden_dim, num_layers, num_heads, attention_dropout, feature_dropout
        )
        self.backward_gat = DirectionalGATStack(
            hidden_dim, num_layers, num_heads, attention_dropout, feature_dropout
        )
        self.direction_fusion = nn.Linear(2 * hidden_dim, hidden_dim, bias=True)

    def forward(
        self,
        node_item_ids,
        forward_edge_index,
        forward_edge_attr,
        backward_edge_index,
        backward_edge_attr,
    ):
        node_features = self.embedding(node_item_ids)
        forward_features = self.forward_gat(
            node_features, forward_edge_index, forward_edge_attr
        )
        backward_features = self.backward_gat(
            node_features, backward_edge_index, backward_edge_attr
        )
        return self.direction_fusion(
            torch.cat([forward_features, backward_features], dim=-1)
        )

## GAT + SR-GNN-Style Readout

In [11]:
class GATSRGNN(nn.Module):
    def __init__(
        self,
        num_items,
        hidden_dim=100,
        num_layers=1,
        num_heads=4,
        attention_dropout=0.05,
        feature_dropout=0.1,
    ):
        super().__init__()
        self.encoder = BidirectionalGATEncoder(
            num_items, hidden_dim, num_layers, num_heads, attention_dropout, feature_dropout
        )
        self.hidden_dim = hidden_dim

        self.sequence_attention_projection = nn.Linear(
            hidden_dim, hidden_dim, bias=False
        )
        self.last_click_attention_projection = nn.Linear(
            hidden_dim, hidden_dim, bias=False
        )
        self.attention_score_projection = nn.Linear(hidden_dim, 1, bias=False)
        self.hybrid_projection = nn.Linear(2 * hidden_dim, hidden_dim, bias=True)
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1.0 / math.sqrt(self.hidden_dim)
        for parameter in self.parameters():
            parameter.data.uniform_(-stdv, stdv)
        with torch.no_grad():
            self.encoder.embedding.weight[0].fill_(0)

    def forward(self, batch):
        """Return logits [batch_size, num_items]."""
        node_hidden = self.encoder(
            batch.x,
            batch.forward_edge_index,
            batch.forward_edge_attr,
            batch.backward_edge_index,
            batch.backward_edge_attr,
        )

        sequence_hidden, sequence_batch = self._sequence_hidden(batch, node_hidden)
        sequence_lengths = batch.sequence_length.view(-1).long()
        sequence_offsets = torch.cat(
            [
                sequence_lengths.new_zeros(1),
                sequence_lengths.cumsum(dim=0)[:-1],
            ]
        )

        last_sequence_index = sequence_offsets + sequence_lengths - 1
        local_preference = sequence_hidden[last_sequence_index]

        expanded_local_preference = local_preference[sequence_batch]
        attention_logits = self.attention_score_projection(
            torch.sigmoid(
                self.sequence_attention_projection(sequence_hidden)
                + self.last_click_attention_projection(expanded_local_preference)
            )
        ).squeeze(-1)
        attention_weights = pyg_softmax(attention_logits, sequence_batch)
        attention_weights = attention_weights.to(sequence_hidden.dtype)

        global_preference = torch.zeros_like(local_preference)
        global_preference.scatter_add_(
            0,
            sequence_batch.unsqueeze(-1).expand_as(sequence_hidden),
            attention_weights.unsqueeze(-1) * sequence_hidden,
        )

        session_representation = self.hybrid_projection(
            torch.cat([local_preference, global_preference], dim=-1)
        )

        item_embeddings = self.encoder.embedding.weight
        logits = session_representation @ item_embeddings.T
        return logits

    @staticmethod
    def _sequence_hidden(batch, node_hidden):
        """Map contextual node embeddings back to original prefix positions."""
        sequence_lengths = batch.sequence_length.view(-1).long()
        sequence_parts = torch.split(batch.sequence, sequence_lengths.tolist())

        hidden_parts = []
        batch_parts = []
        for graph_index, local_sequence in enumerate(sequence_parts):
            node_offset = batch.ptr[graph_index]
            hidden_parts.append(node_hidden[node_offset + local_sequence])
            batch_parts.append(
                local_sequence.new_full((local_sequence.numel(),), graph_index)
            )

        return torch.cat(hidden_parts, dim=0), torch.cat(batch_parts, dim=0)

In [12]:
def run_gat_graph_feature_checks():
    single = build_session_graph([1], 2)
    assert single.forward_edge_attr.shape == (0, 3)
    assert single.backward_edge_attr.shape == (0, 3)

    repeated = build_session_graph([1, 2, 1, 2], 3)
    assert repeated.forward_edge_attr.shape == (2, 3)
    assert repeated.backward_edge_attr.shape == (2, 3)
    assert torch.all(repeated.forward_edge_attr[:, 0] > 0)
    assert torch.all((0 <= repeated.forward_edge_attr[:, 1]) & (repeated.forward_edge_attr[:, 1] <= 1))
    assert torch.all((0 <= repeated.forward_edge_attr[:, 2]) & (repeated.forward_edge_attr[:, 2] <= 1))


def run_gat_sr_gnn_forward_smoke_test(device=torch.device("cpu")):
    rows = [([1], 2), ([1, 2, 1, 2], 3)]
    loader = PyGDataLoader(SessionGraphDataset(rows), batch_size=2, shuffle=False)
    batch = next(iter(loader)).to(device)
    model = GATSRGNN(num_items=8).to(device)
    logits = model(batch)
    assert logits.shape == (len(rows), 8)
    assert torch.isfinite(logits).all()

## Training and Evaluation Protocol

The SR-GNN paper evaluates with `P@20` and `MRR@20`. Its reported setup uses hidden size `100`, a random `10%` validation split from the training set, Adam with learning rate `0.001`, learning-rate decay by `0.1` every 3 epochs, batch size `100`, and L2 penalty `1e-5`.

This notebook uses that setup directly for the GAT + SR-GNN-style readout experiment.

In [13]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def random_train_validation_split(rows, validation_fraction=0.1, seed=42):
    indices = list(range(len(rows)))
    random.Random(seed).shuffle(indices)
    validation_size = max(1, int(len(indices) * validation_fraction))
    validation_indices = set(indices[:validation_size])

    train_rows = []
    validation_rows = []
    for index, row in enumerate(rows):
        if index in validation_indices:
            validation_rows.append(row)
        else:
            train_rows.append(row)
    return train_rows, validation_rows


def build_loader(rows, batch_size, shuffle, device, config):
    dataset = SessionGraphDataset(rows)
    use_cuda = device.type == "cuda"
    num_workers = int(config.get("num_workers", 0)) if use_cuda else 0
    loader_kwargs = {
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": num_workers,
        "pin_memory": bool(config.get("pin_memory", use_cuda)) and use_cuda,
        "persistent_workers": num_workers > 0,
    }
    if num_workers > 0:
        loader_kwargs["prefetch_factor"] = int(config.get("prefetch_factor", 2))
    return PyGDataLoader(dataset, **loader_kwargs)


def mask_padding_item(logits):
    logits = logits.clone()
    logits[:, 0] = -torch.finfo(logits.dtype).max
    return logits


def log_cuda_memory(prefix):
    if torch.cuda.is_available():
        allocated = torch.cuda.max_memory_allocated() / 1024**3
        reserved = torch.cuda.max_memory_reserved() / 1024**3
        print(f"{prefix}_cuda_peak_allocated_gb={allocated:.2f}")
        print(f"{prefix}_cuda_peak_reserved_gb={reserved:.2f}")


In [14]:
def precision_mrr_at_k(logits, targets, k=20):
    k = min(k, logits.size(1))
    top_items = logits.topk(k, dim=1).indices
    matches = top_items.eq(targets.view(-1, 1))

    hits = matches.any(dim=1).float()
    ranks = torch.zeros(targets.size(0), device=logits.device)
    matched_rows, matched_cols = matches.nonzero(as_tuple=True)
    ranks[matched_rows] = matched_cols.float() + 1
    reciprocal_ranks = torch.where(ranks > 0, 1.0 / ranks, torch.zeros_like(ranks))

    return hits.sum().item(), reciprocal_ranks.sum().item(), targets.size(0)


@torch.no_grad()
def evaluate(model, loader, device, k=20):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    total_hits = 0.0
    total_mrr = 0.0

    for batch_index, batch in enumerate(loader):
        batch = batch.to(device, non_blocking=device.type == "cuda")
        logits = mask_padding_item(model(batch))
        loss = cross_entropy_loss(logits, batch.y, config)
        assert_finite("evaluation logits", logits, batch_index)
        assert_finite("evaluation loss", loss, batch_index)

        hits, mrr, examples = precision_mrr_at_k(logits, batch.y, k=k)
        total_loss += loss.item() * examples
        total_examples += examples
        total_hits += hits
        total_mrr += mrr

    return {
        "loss": total_loss / total_examples,
        "precision@20": 100.0 * total_hits / total_examples,
        "mrr@20": 100.0 * total_mrr / total_examples,
        "examples": total_examples,
    }


def assert_finite(name, tensor, batch_index):
    if not torch.isfinite(tensor).all():
        raise RuntimeError(f"Non-finite {name} detected in batch {batch_index}")


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total_examples = 0

    for batch_index, batch in enumerate(loader):
        batch = batch.to(device, non_blocking=device.type == "cuda")
        optimizer.zero_grad(set_to_none=True)
        logits = mask_padding_item(model(batch))
        loss = cross_entropy_loss(logits, batch.y, config)
        assert_finite("logits", logits, batch_index)
        assert_finite("loss", loss, batch_index)

        loss.backward()
        optimizer.step()

        examples = batch.y.size(0)
        total_loss += loss.item() * examples
        total_examples += examples

    return total_loss / total_examples

def cross_entropy_loss(logits, targets, config):
    # Exclude padding item 0 from the loss. With label smoothing, every class
    # contributes to the objective; the masked padding logit is -finfo.max,
    # which makes the smoothed padding term non-finite.
    return F.cross_entropy(
        logits[:, 1:],
        targets - 1,
        label_smoothing=config.get("label_smoothing", 0.0),
    )


def build_optimizer(model, config):
    optimizer_name = config.get("optimizer", "adam").lower()
    if optimizer_name == "adamw":
        embedding_weight_decay = config.get("embedding_weight_decay", 0.0)
        decay_params = []
        embedding_params = []
        no_decay_params = []
        for name, parameter in model.named_parameters():
            if not parameter.requires_grad:
                continue
            if "embedding" in name:
                embedding_params.append(parameter)
            elif name.endswith("bias"):
                no_decay_params.append(parameter)
            else:
                decay_params.append(parameter)
        return torch.optim.AdamW(
            [
                {"params": decay_params, "weight_decay": config["weight_decay"]},
                {"params": embedding_params, "weight_decay": embedding_weight_decay},
                {"params": no_decay_params, "weight_decay": 0.0},
            ],
            lr=config["learning_rate"],
        )
    if optimizer_name != "adam":
        raise ValueError(f"Unsupported optimizer: {config['optimizer']}")
    return torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )


In [15]:
DATASETS = {
    "Yoochoose 1/64": {
        "train_rows": YOOCHOOSE_1_64_TRAIN_ROWS,
        "test_rows": YOOCHOOSE_1_64_TEST_ROWS,
        "num_items": yoochoose_1_64_num_items,
    },
    "Diginetica": {
        "train_rows": DIGINETICA_TRAIN_ROWS,
        "test_rows": DIGINETICA_TEST_ROWS,
        "num_items": diginetica_num_items,
    },
}

REGULARIZATION_PRESETS = {
    "r0_restored_baseline": {
        "attention_dropout": 0.1,
        "feature_dropout": 0.1,
        "optimizer": "adam",
        "weight_decay": 1e-5,
        "embedding_weight_decay": None,
        "label_smoothing": 0.0,
    },
    "r1_mild_dropout": {
        "attention_dropout": 0.15,
        "feature_dropout": 0.15,
        "optimizer": "adam",
        "weight_decay": 1e-5,
        "embedding_weight_decay": None,
        "label_smoothing": 0.0,
    },
    "r2_moderate_dropout": {
        "attention_dropout": 0.2,
        "feature_dropout": 0.2,
        "optimizer": "adam",
        "weight_decay": 1e-5,
        "embedding_weight_decay": None,
        "label_smoothing": 0.0,
    },
    "r3_adamw_decoupled": {
        "attention_dropout": 0.15,
        "feature_dropout": 0.15,
        "optimizer": "adamw",
        "weight_decay": 3e-5,
        "embedding_weight_decay": 0.0,
        "label_smoothing": 0.0,
    },
    "r4_label_smoothing": {
        "attention_dropout": 0.1,
        "feature_dropout": 0.1,
        "optimizer": "adam",
        "weight_decay": 1e-5,
        "embedding_weight_decay": None,
        "label_smoothing": 0.02,
    },
    "r5_restored_baseline_label_smoothing": {
        "attention_dropout": 0.05,
        "feature_dropout": 0.1,
        "optimizer": "adam",
        "weight_decay": 1e-5,
        "embedding_weight_decay": None,
        "label_smoothing": 0.01,
    },
}

regularization_variant = "r5_restored_baseline_label_smoothing"
regularization_config = REGULARIZATION_PRESETS[regularization_variant]

config = {
    "experiment_name": "iteration_10_gat_edge_attr_dropout_split",
    "accelerator": "gpu_t4_x2_single_gpu",
    "architecture_source_commit": "7bff12dc33875b37e78bd8d15ae96963673a899c",
    "regularization_variant": regularization_variant,
    "epochs": 30,
    "patience": 5,
    "batch_size": 100,
    "num_workers": 2,
    "prefetch_factor": 2,
    "pin_memory": True,
    "learning_rate": 0.001,
    "lr_decay_step": 3,
    "lr_decay_gamma": 0.1,
    "validation_fraction": 0.1,
    "hidden_dim": 100,
    "num_layers": 1,
    "num_heads": 4,
    "seed": 42,
    **regularization_config,
}

device = get_device()
print(f"device={device}")
print(config)

device=cuda
{'experiment_name': 'iteration_10_gat_edge_attr_dropout_split', 'accelerator': 'gpu_t4_x2_single_gpu', 'architecture_source_commit': '7bff12dc33875b37e78bd8d15ae96963673a899c', 'regularization_variant': 'r5_restored_baseline_label_smoothing', 'epochs': 30, 'patience': 5, 'batch_size': 100, 'num_workers': 2, 'prefetch_factor': 2, 'pin_memory': True, 'learning_rate': 0.001, 'lr_decay_step': 3, 'lr_decay_gamma': 0.1, 'validation_fraction': 0.1, 'hidden_dim': 100, 'num_layers': 1, 'num_heads': 4, 'seed': 42, 'attention_dropout': 0.05, 'feature_dropout': 0.1, 'optimizer': 'adam', 'weight_decay': 1e-05, 'embedding_weight_decay': None, 'label_smoothing': 0.01}


In [16]:
def checkpoint_name(dataset_name):
    safe_name = dataset_name.lower().replace(" ", "_").replace("/", "_")
    return f"gat_sr_gnn_{safe_name}.pt"


def run_dataset_experiment(dataset_name, dataset, config, device):
    set_seed(config["seed"])
    train_rows = list(dataset["train_rows"])
    test_rows = list(dataset["test_rows"])
    train_rows, validation_rows = random_train_validation_split(
        train_rows,
        validation_fraction=config["validation_fraction"],
        seed=config["seed"],
    )

    train_loader = build_loader(
        train_rows,
        config["batch_size"],
        shuffle=True,
        device=device,
        config=config,
    )
    validation_loader = build_loader(
        validation_rows,
        config["batch_size"],
        shuffle=False,
        device=device,
        config=config,
    )
    test_loader = build_loader(
        test_rows,
        config["batch_size"],
        shuffle=False,
        device=device,
        config=config,
    )

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    model = GATSRGNN(
        num_items=dataset["num_items"],
        hidden_dim=config["hidden_dim"],
        num_layers=config["num_layers"],
        num_heads=config["num_heads"],
        attention_dropout=config["attention_dropout"],
        feature_dropout=config["feature_dropout"],
    ).to(device)
    optimizer = build_optimizer(model, config)
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=config["lr_decay_step"],
        gamma=config["lr_decay_gamma"],
    )
    history = []
    best_validation_mrr = -1.0
    best_validation_precision = -1.0
    best_epoch = 0
    best_state = None
    bad_counter = 0

    for epoch in range(1, config["epochs"] + 1):
        started_at = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        validation_metrics = evaluate(model, validation_loader, device)
        scheduler.step()

        row = {
            "dataset": dataset_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": validation_metrics["loss"],
            "validation_precision@20": validation_metrics["precision@20"],
            "validation_mrr@20": validation_metrics["mrr@20"],
            "epoch_seconds": time.time() - started_at,
            "train_examples": len(train_rows),
            "validation_examples": len(validation_rows),
            "test_examples": len(test_rows),
        }
        history.append(row)
        print(
            f"{dataset_name} epoch {epoch:02d} "
            f"loss={train_loss:.4f} "
            f"val_loss={row['validation_loss']:.4f} "
            f"val_P@20={row['validation_precision@20']:.2f} "
            f"val_MRR@20={row['validation_mrr@20']:.2f} "
            f"time={row['epoch_seconds']:.1f}s"
        )

        if validation_metrics["mrr@20"] >= best_validation_mrr:
            best_validation_mrr = validation_metrics["mrr@20"]
            best_validation_precision = validation_metrics["precision@20"]
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            bad_counter = 0
        else:
            bad_counter += 1
            if bad_counter >= config["patience"]:
                print(f"early stopping at epoch {epoch}; best epoch was {best_epoch}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    test_metrics = evaluate(model, test_loader, device)
    log_cuda_memory(dataset_name)

    checkpoint_path = CHECKPOINTS_DIR / checkpoint_name(dataset_name)
    torch.save(
        {
            "dataset": dataset_name,
            "model": "GATSRGNN",
            "config": dict(config),
            "num_items": dataset["num_items"],
            "state_dict": model.state_dict(),
            "test_metrics": test_metrics,
            "history": history,
            "best_epoch": best_epoch,
            "best_validation_precision@20": best_validation_precision,
            "best_validation_mrr@20": best_validation_mrr,
        },
        checkpoint_path,
    )

    result = {
        "dataset": dataset_name,
        "test_precision@20": test_metrics["precision@20"],
        "test_mrr@20": test_metrics["mrr@20"],
        "test_loss": test_metrics["loss"],
        "best_epoch": best_epoch,
        "best_validation_precision@20": best_validation_precision,
        "best_validation_mrr@20": best_validation_mrr,
        "train_examples": len(train_rows),
        "validation_examples": len(validation_rows),
        "test_examples": len(test_rows),
        "num_items": dataset["num_items"],
        "checkpoint_path": str(checkpoint_path),
    }
    del model, optimizer, scheduler, train_loader, validation_loader, test_loader
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return result, history

In [17]:
all_results = []
all_history = []

for dataset_name, dataset in DATASETS.items():
    result, history = run_dataset_experiment(dataset_name, dataset, config, device)
    all_results.append(result)
    all_history.extend(history)

results = pd.DataFrame(all_results)
history = pd.DataFrame(all_history)

results_path = RESULTS_DIR / "gat_sr_gnn_results.csv"
history_path = RESULTS_DIR / "gat_sr_gnn_history.csv"
results.to_csv(results_path, index=False)
history.to_csv(history_path, index=False)

display(results)
print(f"saved {results_path}")
print(f"saved {history_path}")

Yoochoose 1/64 epoch 01 loss=5.5989 val_loss=4.8283 val_P@20=65.03 val_MRR@20=28.51 time=176.5s
Yoochoose 1/64 epoch 02 loss=4.5562 val_loss=4.6065 val_P@20=67.68 val_MRR@20=30.10 time=191.7s
Yoochoose 1/64 epoch 03 loss=4.3063 val_loss=4.5443 val_P@20=68.59 val_MRR@20=30.75 time=184.4s
Yoochoose 1/64 epoch 04 loss=3.8860 val_loss=4.4385 val_P@20=69.86 val_MRR@20=32.51 time=172.0s
Yoochoose 1/64 epoch 05 loss=3.7948 val_loss=4.4460 val_P@20=69.83 val_MRR@20=32.66 time=175.6s
Yoochoose 1/64 epoch 06 loss=3.7474 val_loss=4.4616 val_P@20=69.77 val_MRR@20=32.69 time=171.7s
Yoochoose 1/64 epoch 07 loss=3.6695 val_loss=4.4644 val_P@20=69.79 val_MRR@20=32.81 time=171.5s
Yoochoose 1/64 epoch 08 loss=3.6625 val_loss=4.4678 val_P@20=69.75 val_MRR@20=32.82 time=192.3s
Yoochoose 1/64 epoch 09 loss=3.6570 val_loss=4.4713 val_P@20=69.72 val_MRR@20=32.80 time=173.1s
Yoochoose 1/64 epoch 10 loss=3.6480 val_loss=4.4717 val_P@20=69.71 val_MRR@20=32.79 time=173.2s
Yoochoose 1/64 epoch 11 loss=3.6474 val_

,dataset,test_precision@20,test_mrr@20,test_loss,best_epoch,best_validation_precision@20,best_validation_mrr@20,train_examples,validation_examples,test_examples,num_items,checkpoint_path
0,Yoochoose 1/64,70.047229,31.096935,4.440568,8,69.752602,32.817845,332874,36985,55898,37484,/kaggle/working/results/checkpoints/gat_sr_gnn...
1,Diginetica,50.427224,16.688020,5.647139,4,55.654857,19.532865,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sr_gnn...


saved /kaggle/working/results/gat_sr_gnn_results.csv
saved /kaggle/working/results/gat_sr_gnn_history.csv


In [18]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

results.plot.bar(
    x="dataset",
    y="test_precision@20",
    ax=axes[0],
    legend=False,
    color="#3b6ea8",
    title="Test Precision@20",
)
axes[0].set_ylabel("%")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=25)

results.plot.bar(
    x="dataset",
    y="test_mrr@20",
    ax=axes[1],
    legend=False,
    color="#b45f3c",
    title="Test MRR@20",
)
axes[1].set_ylabel("%")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
figure_path = RESULTS_DIR / "gat_sr_gnn_metrics.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
print(f"saved {figure_path}")

saved /kaggle/working/results/gat_sr_gnn_metrics.png
